# Dashboard Dataset Preparation

This notebook prepares the final analytical dataset for the Power BI dashboard.

The dataset combines the cleaned HR employee attrition data with the
finalized rule-based employee risk segmentation developed in the
employee risk segmentation analysis.

The notebook does not introduce new analytical rules. It reproduces the
final risk indicators and risk segmentation using the same definitions
established during the analytical phase.

The resulting dataset will be used as the primary data source for the
Power BI dashboard.

## 1. Load Clean Dataset

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

file_path = "../data/processed/hr_employee_attrition_clean.csv"

df = pd.read_csv(file_path)

df.shape

(1470, 32)

## 2. Risk Factor Definitions

In [2]:
df["risk_overtime"] = (
    df["over_time"] == "Yes"
).astype(int)

df["risk_low_job_satisfaction"] = (
    df["job_satisfaction"] <= 2
).astype(int)

df["risk_low_environment_satisfaction"] = (
    df["environment_satisfaction"] <= 2
).astype(int)

df["risk_low_job_involvement"] = (
    df["job_involvement"] <= 2
).astype(int)

df["risk_low_job_level"] = (
    df["job_level"] == 1
).astype(int)

df["risk_early_tenure"] = (
    df["years_at_company"] <= 3
).astype(int)

df["risk_low_income"] = (
    df["monthly_income"] <= df["monthly_income"].median()
).astype(int)

In [3]:
risk_factor_columns = [
    "risk_overtime",
    "risk_low_job_satisfaction",
    "risk_low_environment_satisfaction",
    "risk_low_job_involvement",
    "risk_low_job_level",
    "risk_early_tenure",
    "risk_low_income"
]

## 3. Risk Score

In [4]:
df["risk_score"] = df[risk_factor_columns].sum(axis=1)

df["risk_score"].value_counts().sort_index()

risk_score
0    100
1    288
2    375
3    301
4    255
5    113
6     30
7      8
Name: count, dtype: int64

## 4. Risk Segment

In [5]:
def assign_risk_segment(score):
    if score <= 3:
        return "Low Risk"
    elif score == 4:
        return "Medium Risk"
    else:
        return "High Risk"

df["risk_segment"] = df["risk_score"].apply(assign_risk_segment)

In [6]:
risk_segment_summary = (
    df.groupby("risk_segment")
      .agg(
          employee_count=("employee_number", "count"),
          attrition_rate=(
              "attrition",
              lambda x: (x == "Yes").mean() * 100
          )
      )
      .round(2)
)

risk_segment_summary

,employee_count,attrition_rate
risk_segment,,
High Risk,151,50.99
Low Risk,1064,9.02
Medium Risk,255,25.10


## 5. Validate Risk Columns

In [7]:
risk_columns = (
    risk_factor_columns
    + ["risk_score", "risk_segment"]
)

df[risk_columns].head()

,risk_overtime,risk_low_job_satisfaction,risk_low_environment_satisfaction,risk_low_job_involvement,risk_low_job_level,risk_early_tenure,risk_low_income,risk_score,risk_segment
0,1,0,1,0,0,0,0,2,Low Risk
1,0,1,0,1,0,0,0,2,Low Risk
2,1,0,0,1,1,1,1,5,High Risk
3,1,0,0,0,1,0,1,3,Low Risk
4,0,1,1,0,1,1,1,5,High Risk


In [8]:
df[risk_columns].isna().sum()

risk_overtime                        0
risk_low_job_satisfaction            0
risk_low_environment_satisfaction    0
risk_low_job_involvement             0
risk_low_job_level                   0
risk_early_tenure                    0
risk_low_income                      0
risk_score                           0
risk_segment                         0
dtype: int64

## 6. Validate Dataset

In [9]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("Duplicate rows:", df.duplicated().sum())

Rows: 1470
Columns: 41
Duplicate rows: 0


## 7. Final Column Check

In [10]:
df.columns.tolist()

['age',
 'attrition',
 'business_travel',
 'daily_rate',
 'department',
 'distance_from_home',
 'education',
 'education_field',
 'employee_number',
 'environment_satisfaction',
 'gender',
 'hourly_rate',
 'job_involvement',
 'job_level',
 'job_role',
 'job_satisfaction',
 'marital_status',
 'monthly_income',
 'monthly_rate',
 'num_companies_worked',
 'over_time',
 'percent_salary_hike',
 'performance_rating',
 'relationship_satisfaction',
 'stock_option_level',
 'total_working_years',
 'training_times_last_year',
 'work_life_balance',
 'years_at_company',
 'years_in_current_role',
 'years_since_last_promotion',
 'years_with_curr_manager',
 'risk_overtime',
 'risk_low_job_satisfaction',
 'risk_low_environment_satisfaction',
 'risk_low_job_involvement',
 'risk_low_job_level',
 'risk_early_tenure',
 'risk_low_income',
 'risk_score',
 'risk_segment']

## 8. Export

In [11]:
output_path = "../data/processed/hr_employee_attrition_analytics.csv"

df.to_csv(
    output_path,
    index=False
)

print(f"Dashboard dataset saved to: {output_path}")

Dashboard dataset saved to: ../data/processed/hr_employee_attrition_analytics.csv


## 9. Final Verification

In [12]:
dashboard_df = pd.read_csv(output_path)

print("Shape:", dashboard_df.shape)
print("Duplicates:", dashboard_df.duplicated().sum())

dashboard_df.head()

Shape: (1470, 41)
Duplicates: 0


,age,attrition,business_travel,daily_rate,department,distance_from_home,education,education_field,employee_number,environment_satisfaction,gender,hourly_rate,job_involvement,job_level,job_role,job_satisfaction,marital_status,monthly_income,monthly_rate,num_companies_worked,over_time,percent_salary_hike,performance_rating,relationship_satisfaction,stock_option_level,total_working_years,training_times_last_year,work_life_balance,years_at_company,years_in_current_role,years_since_last_promotion,years_with_curr_manager,risk_overtime,risk_low_job_satisfaction,risk_low_environment_satisfaction,risk_low_job_involvement,risk_low_job_level,risk_early_tenure,risk_low_income,risk_score,risk_segment
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,2,Female,94,3,2,Sales Executive,4,Single,5993,19479,8,Yes,11,3,1,0,8,0,1,6,4,0,5,1,0,1,0,0,0,0,2,Low Risk
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,2,3,Male,61,2,2,Research Scientist,2,Married,5130,24907,1,No,23,4,4,1,10,3,3,10,7,1,7,0,1,0,1,0,0,0,2,Low Risk
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,4,4,Male,92,2,1,Laboratory Technician,3,Single,2090,2396,6,Yes,15,3,2,0,7,3,3,0,0,0,0,1,0,0,1,1,1,1,5,High Risk
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,5,4,Female,56,3,1,Research Scientist,3,Married,2909,23159,1,Yes,11,3,3,0,8,3,3,8,7,3,0,1,0,0,0,1,0,1,3,Low Risk
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,7,1,Male,40,3,1,Laboratory Technician,2,Married,3468,16632,9,No,12,3,4,1,6,3,3,2,2,2,2,0,1,1,0,1,1,1,5,High Risk


In [13]:
dashboard_df["risk_segment"].value_counts()

risk_segment
Low Risk       1064
Medium Risk     255
High Risk       151
Name: count, dtype: int64

In [14]:
dashboard_df["risk_score"].value_counts().sort_index()

risk_score
0    100
1    288
2    375
3    301
4    255
5    113
6     30
7      8
Name: count, dtype: int64